# Hands-on Week 11

# Agentic Information Extraction

In this notebook we build a small **agentic information extraction system** without LangChain, AutoGen, or any other agent framework.

The goal is to understand the core ideas behind agentic systems:

- state
- tools
- planning
- retrieval
- observation
- reflection
- validation
- structured output

A traditional RAG pipeline usually performs retrieval once and then generates an answer.

An agentic system can decide whether it has enough information, use tools repeatedly, update its internal state, and stop only when the task is complete.


## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain the difference between a fixed workflow and an agentic workflow.
2. Define an agent state.
3. Implement simple tools for Information Extraction.
4. Build a reasoning loop.
5. Detect missing fields.
6. Validate structured output.
7. Understand why agentic IE can be useful for complex documents.


In [ ]:
import re
import json
from typing import Dict, List, Optional, Any

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from pydantic import BaseModel, ValidationError
    PYDANTIC_AVAILABLE = True
except ImportError:
    PYDANTIC_AVAILABLE = False

def TODO(message: str = "Fill the blank"):
    raise ValueError(message)

np.random.seed(42)


## 1. Mini Document Collection

We use small OCR-like receipt documents.

In a real system, these texts would come from scanned PDFs, OCR, or a document parser.

The task is to extract structured fields:

- merchant
- invoice number
- date
- total
- payment method


In [ ]:
documents = [
    {
        "id": "receipt_001",
        "text": """
        Merchant: Starbucks Hamburg
        Invoice Number: INV-1001
        Date: 2024-03-18
        Items: Latte 4.50 EUR, Sandwich 8.00 EUR, Cookie 2.00 EUR
        Total: 18.50 EUR
        Payment Method: Credit Card
        """,
        "ground_truth": {
            "merchant": "Starbucks Hamburg",
            "invoice_number": "INV-1001",
            "date": "2024-03-18",
            "total": "18.50 EUR",
            "payment_method": "Credit Card"
        }
    },
    {
        "id": "receipt_002",
        "text": """
        Merchant: MediaMarkt Hamburg
        Receipt No: MM-8891
        Purchase Date: 2024-05-11
        Product: Wireless Headphones
        Amount Due: 579.99 EUR
        Paid by: VISA
        """,
        "ground_truth": {
            "merchant": "MediaMarkt Hamburg",
            "invoice_number": "MM-8891",
            "date": "2024-05-11",
            "total": "579.99 EUR",
            "payment_method": "VISA"
        }
    },
    {
        "id": "receipt_003",
        "text": """
        IKEA Hamburg
        Document ID: IK-2230
        Date of Sale: 2024-04-02
        Chair 49.99 EUR
        Table 89.99 EUR
        Lamp 19.99 EUR
        Final Total 159.97 EUR
        """,
        "ground_truth": {
            "merchant": "IKEA Hamburg",
            "invoice_number": "IK-2230",
            "date": "2024-04-02",
            "total": "159.97 EUR",
            "payment_method": None
        }
    },
    {
        "id": "receipt_004",
        "text": """
        REWE Markt GmbH
        Transaction: RW-7721
        2024/06/09
        Groceries and household products
        Sum: 42.35 EUR
        Payment: Girocard
        """,
        "ground_truth": {
            "merchant": "REWE Markt GmbH",
            "invoice_number": "RW-7721",
            "date": "2024/06/09",
            "total": "42.35 EUR",
            "payment_method": "Girocard"
        }
    }
]

pd.DataFrame([
    {"id": doc["id"], "text": doc["text"].strip()[:100] + "...", **doc["ground_truth"]}
    for doc in documents
])


## 2. Traditional RAG vs Agentic IE

A simple RAG pipeline looks like this:

```text
query → retrieve top-k chunks → prompt LLM → answer
```

An agentic IE workflow looks like this:

```text
goal → inspect state → choose tool → observe result → update state → reflect → continue or stop
```

The key difference is **iteration**.


## 3. Chunking

To simulate document retrieval, we split documents into smaller pieces.

In a real RAG system, chunking quality strongly affects extraction quality.


In [ ]:
def chunk_text(text: str, chunk_size: int = 120, overlap: int = 30) -> List[str]:
    text = re.sub(r"\s+", " ", text.strip())
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])

        if end == len(text):
            break

        start = end - overlap

    return chunks


all_chunks = []

for doc in documents:
    chunks = chunk_text(doc["text"])
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "doc_id": doc["id"],
            "chunk_id": f"{doc['id']}_chunk_{i}",
            "text": chunk
        })

chunks_df = pd.DataFrame(all_chunks)
chunks_df


## 4. Embeddings and Retrieval Tool

The first tool our agent needs is a retrieval tool.

Given a query, it returns the most relevant chunks.


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in all_chunks]
chunk_embeddings = embedding_model.encode(chunk_texts, convert_to_numpy=True)

print("Number of chunks:", len(chunk_texts))
print("Embedding shape:", chunk_embeddings.shape)


In [ ]:
def retrieve(query: str, top_k: int = 3) -> pd.DataFrame:
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(query_embedding, chunk_embeddings)[0]

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    rows = []
    for rank, idx in enumerate(ranked_indices, start=1):
        rows.append({
            "rank": rank,
            "score": float(scores[idx]),
            "doc_id": all_chunks[idx]["doc_id"],
            "chunk_id": all_chunks[idx]["chunk_id"],
            "text": all_chunks[idx]["text"]
        })

    return pd.DataFrame(rows)

retrieve("Find the total amount and payment method", top_k=4)


## 5. Rule-based Extraction Tool

To keep this notebook executable without an external LLM API, we implement a simple extraction tool using regular expressions.

In a production system, this tool could be replaced by:

- an LLM call
- a fine-tuned IE model
- a layout-aware model
- a vision-language model

The agentic logic remains the same.


In [ ]:
def extract_fields_from_text(text: str) -> Dict[str, Optional[str]]:
    fields = {
        "merchant": None,
        "invoice_number": None,
        "date": None,
        "total": None,
        "payment_method": None
    }

    pattern_groups = {
        "merchant": [
            r"Merchant:\s*([^\n]+)",
            r"^(IKEA Hamburg)",
            r"^(REWE Markt GmbH)",
        ],
        "invoice_number": [
            r"Invoice Number:\s*([A-Z0-9\-]+)",
            r"Receipt No:\s*([A-Z0-9\-]+)",
            r"Document ID:\s*([A-Z0-9\-]+)",
            r"Transaction:\s*([A-Z0-9\-]+)"
        ],
        "date": [
            r"Date:\s*(\d{4}-\d{2}-\d{2})",
            r"Purchase Date:\s*(\d{4}-\d{2}-\d{2})",
            r"Date of Sale:\s*(\d{4}-\d{2}-\d{2})",
            r"(\d{4}/\d{2}/\d{2})"
        ],
        "total": [
            r"Total:\s*([0-9]+\.[0-9]{2}\s*EUR)",
            r"Amount Due:\s*([0-9]+\.[0-9]{2}\s*EUR)",
            r"Final Total\s*([0-9]+\.[0-9]{2}\s*EUR)",
            r"Sum:\s*([0-9]+\.[0-9]{2}\s*EUR)"
        ],
        "payment_method": [
            r"Payment Method:\s*([^\n]+)",
            r"Paid by:\s*([^\n]+)",
            r"Payment:\s*([^\n]+)"
        ]
    }

    for field, patterns in pattern_groups.items():
        for pattern in patterns:
            match = re.search(pattern, text.strip(), re.MULTILINE)
            if match:
                fields[field] = match.group(1).strip()
                break

    return fields


extract_fields_from_text(documents[0]["text"])


## 6. Agent State

An agent needs a memory of what it already knows.

We store the current extraction result in a state dictionary.

Fields that are not found yet are set to `None`.


In [ ]:
def initialize_state(document_id: str) -> Dict[str, Any]:
    return {
        "document_id": document_id,
        "merchant": None,
        "invoice_number": None,
        "date": None,
        "total": None,
        "payment_method": None,
        "evidence": [],
        "steps": [],
        "done": False
    }

state = initialize_state("receipt_001")
state


## 7. Reflection: What Is Missing?

Reflection means that the agent checks its own current state.

For IE, a simple reflection step is:

> Which required fields are still missing?


In [ ]:
REQUIRED_FIELDS = ["merchant", "invoice_number", "date", "total"]
OPTIONAL_FIELDS = ["payment_method"]

def missing_required_fields(state: Dict[str, Any]) -> List[str]:
    return [field for field in REQUIRED_FIELDS if state.get(field) is None]

missing_required_fields(state)


## 8. Updating State from Observations

When the agent retrieves text and extracts information, it observes new candidate fields.

Then it updates its state.


In [ ]:
def update_state_from_extraction(
    state: Dict[str, Any],
    extracted: Dict[str, Optional[str]],
    evidence_text: str,
    tool_name: str
) -> Dict[str, Any]:

    for field, value in extracted.items():
        if value is not None and state.get(field) is None:
            state[field] = value

    state["evidence"].append(evidence_text)
    state["steps"].append({
        "tool": tool_name,
        "extracted": extracted,
        "missing_after_step": missing_required_fields(state)
    })

    if len(missing_required_fields(state)) == 0:
        state["done"] = True

    return state


## 9. Planner: Choosing the Next Action

A simple planner chooses what the agent should do next.

This is not an LLM planner. It is a transparent rule-based planner designed for teaching.

In real agentic systems, the planning step may be performed by an LLM.


In [ ]:
def plan_next_action(state: Dict[str, Any]) -> Dict[str, str]:
    missing = missing_required_fields(state)

    if not missing:
        return {"action": "finish", "query": ""}

    next_field = missing[0]

    query_templates = {
        "merchant": "Find the merchant or company name",
        "invoice_number": "Find the invoice number receipt number document ID or transaction number",
        "date": "Find the receipt date purchase date or date of sale",
        "total": "Find the total amount amount due final total or sum"
    }

    return {"action": "retrieve", "query": query_templates[next_field]}

plan_next_action(state)


## 10. Agent Loop

Now we combine everything:

1. Inspect current state.
2. Plan next action.
3. Use a tool.
4. Observe result.
5. Update state.
6. Reflect.
7. Stop when all required fields are found.


In [ ]:
def run_agent(document_id: str, max_steps: int = 6, verbose: bool = True) -> Dict[str, Any]:
    state = initialize_state(document_id)

    for step in range(max_steps):
        if verbose:
            print(f"\n--- Step {step + 1} ---")
            print("Current missing fields:", missing_required_fields(state))

        action = plan_next_action(state)

        if action["action"] == "finish":
            if verbose:
                print("All required fields found. Finishing.")
            break

        query = action["query"] + f" for document {document_id}"
        retrieved = retrieve(query, top_k=5)

        retrieved_current_doc = retrieved[retrieved["doc_id"] == document_id]

        if len(retrieved_current_doc) == 0:
            observation = retrieved.iloc[0]["text"]
        else:
            observation = retrieved_current_doc.iloc[0]["text"]

        extracted = extract_fields_from_text(observation)

        if verbose:
            print("Action:", action["action"])
            print("Query:", query)
            print("Observation:", observation)
            print("Extracted:", extracted)

        state = update_state_from_extraction(
            state=state,
            extracted=extracted,
            evidence_text=observation,
            tool_name="retrieve_and_extract"
        )

        if state["done"]:
            if verbose:
                print("All required fields found. Finishing.")
            break

    return state


agent_result = run_agent("receipt_001")
agent_result


## 11. Display Agent Trace

Agent traces are important for debugging.

They show what the system did at each step.


In [ ]:
def show_agent_trace(state: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    for i, step in enumerate(state["steps"], start=1):
        row = {
            "step": i,
            "tool": step["tool"],
            "missing_after_step": step["missing_after_step"]
        }
        row.update(step["extracted"])
        rows.append(row)

    return pd.DataFrame(rows)

show_agent_trace(agent_result)


## 12. Structured Output with Pydantic

After extraction, we want a validated structured object.

Pydantic is useful because it checks whether the output matches the expected schema.


In [ ]:
if PYDANTIC_AVAILABLE:
    class ReceiptExtraction(BaseModel):
        merchant: str
        invoice_number: str
        date: str
        total: str
        payment_method: Optional[str] = None

    try:
        validated = ReceiptExtraction(
            merchant=agent_result["merchant"],
            invoice_number=agent_result["invoice_number"],
            date=agent_result["date"],
            total=agent_result["total"],
            payment_method=agent_result["payment_method"]
        )
        print(validated.model_dump_json(indent=2))
    except ValidationError as e:
        print(e)
else:
    print("Pydantic is not installed. Install it with: pip install pydantic")


## 13. Evaluation

Now we run the agent on all receipts and compare predictions with the ground truth.


In [ ]:
def extract_prediction_from_state(state: Dict[str, Any]) -> Dict[str, Optional[str]]:
    return {
        "merchant": state["merchant"],
        "invoice_number": state["invoice_number"],
        "date": state["date"],
        "total": state["total"],
        "payment_method": state["payment_method"]
    }

evaluation_rows = []

for doc in documents:
    state = run_agent(doc["id"], verbose=False)
    prediction = extract_prediction_from_state(state)
    ground_truth = doc["ground_truth"]

    for field in ["merchant", "invoice_number", "date", "total", "payment_method"]:
        evaluation_rows.append({
            "doc_id": doc["id"],
            "field": field,
            "prediction": prediction[field],
            "ground_truth": ground_truth[field],
            "correct": prediction[field] == ground_truth[field]
        })

eval_df = pd.DataFrame(evaluation_rows)
eval_df


In [ ]:
field_accuracy = eval_df.groupby("field")["correct"].mean().reset_index()
field_accuracy


## 14. Failure Case: Missing Information

A good agent should not hallucinate missing information.

For `receipt_003`, the payment method is not provided.

The system should leave it as `None`.


In [ ]:
state_003 = run_agent("receipt_003", verbose=True)

print("\nFinal state:")
print(json.dumps(extract_prediction_from_state(state_003), indent=2))


## 15. Exercise 1 — Add a New Tool

Add a tool that searches only inside one document instead of all chunks.

Suggested function:

```python
def lookup_document(document_id):
    ...
```

Then modify the agent so that it can use this tool when retrieval fails.


In [ ]:
def lookup_document(document_id: str) -> str:
    # TODO:
    # Return the full text of the document with the matching document_id.
    # Hint: loop over documents and compare doc["id"].
    TODO("Implement lookup_document")

# lookup_document("receipt_001")


## 16. Exercise 2 — Improve the Planner

The current planner retrieves one missing field at a time.

Improve it so that it can retrieve multiple fields together.

Example:

If both `date` and `total` are missing, retrieve:

```text
Find the date and total amount
```


In [ ]:
def improved_plan_next_action(state: Dict[str, Any]) -> Dict[str, str]:
    missing = missing_required_fields(state)

    # TODO:
    # If no fields are missing, return finish.
    # If multiple fields are missing, create a combined retrieval query.
    # Otherwise retrieve the single missing field.

    TODO("Implement improved planner")


## 17. Exercise 3 — Add Validation Rules

Add simple validation rules:

- total should contain `EUR`
- invoice number should contain at least one digit
- date should contain a year

Return a list of validation errors.


In [ ]:
def validate_extraction(prediction: Dict[str, Optional[str]]) -> List[str]:
    errors = []

    # TODO:
    # Add validation rules here.
    # Example:
    # if prediction["total"] is not None and "EUR" not in prediction["total"]:
    #     errors.append("Total does not contain currency.")

    return errors

validate_extraction(extract_prediction_from_state(agent_result))


## 18. Exercise 4 — RAG vs Agent

Complete the table conceptually.

| Dimension | Traditional RAG | Agentic IE |
|---|---|---|
| Retrieval | ? | ? |
| Planning | ? | ? |
| Iteration | ? | ? |
| Validation | ? | ? |
| Failure Recovery | ? | ? |


## 19. Reflection Questions

Answer briefly:

1. What makes this system agentic?
2. How is the agent state different from a normal function output?
3. Why is tool use important?
4. Why is reflection useful?
5. What could go wrong if the planner chooses a bad query?
6. How would an LLM-based planner improve this system?
7. What are the limitations of our simple agent?
8. How could this notebook be extended to real PDFs?


## Summary

In this notebook, we built a simple agentic IE system.

Main ideas:

- Agents maintain state.
- Agents use tools.
- Agents observe tool outputs.
- Agents update memory.
- Agents reflect on missing information.
- Agents stop when the goal is achieved.
- Structured validation makes extraction more reliable.

This is the final conceptual step in the course:

```text
Rules → Classical ML → Transformers → LLMs → RAG → Agentic IE
```
